# Figure 2 — Make Panels

Generates abstract Figure 2 panels A–I (MVT, behaviour metrics, manipulations).
Sources: `fig4_general_mvt_results.ipynb`, `fig6_travel_cost_manipulations.ipynb`,
`fig7_global_reward_manipulations.ipynb`

In [ ]:
import sys

ABSTRACT_ROOT = "C:/git/cosyne2026_abstract"
PAPER_CODE    = "C:/git/aind-vr-foraging-paper/code"
sys.path.insert(0, ABSTRACT_ROOT)
sys.path.insert(0, PAPER_CODE)

from pathlib import Path
import math
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy.stats import ttest_rel
from scipy.optimize import brentq
from matplotlib.ticker import FixedLocator

from aind_vr_foraging_analysis.utils import plotting
from aind_vr_foraging_analysis.utils.parsing import data_access
from aind_vr_foraging_analysis.utils.parsing.supplementary_parsing import AddExtraColumns as Add
from tools.style import apply_style, panel_fig, save_panel, PALETTE

color1, color2, color3 = "#d95f02", "#1b9e77", "#7570b3"
color_dict_label = {"90": color1, "60": color2, "0": color3}

S3_ROOT = "s3://aind-scratch-data/vr-foraging/vr-foraging-manuscript"
STORAGE_OPTIONS = {"skip_signature": "true"}
STAGES_CSV = "C:/git/aind-vr-foraging-paper/code/notebooks/results/sessions_stages.csv"

apply_style()
PANELS_DIR = Path(ABSTRACT_ROOT) / "figures" / "fig2" / "panels"
PANELS_DIR.mkdir(exist_ok=True)
print("Panels will be saved to:", PANELS_DIR)


## Data loading

Three separate cohorts with different exclusion sets and stage filters.

In [ ]:
# --- Data loading for three cohorts ---

# Cohort 1: fig4 (base behaviour)
EXCLUDED_FIG4 = {713578, 754575, 715866}
_, _, macro4 = data_access._load_session_catalogue_and_sites(
    S3_ROOT, STORAGE_OPTIONS, EXCLUDED_FIG4
)
macro4 = data_access.prepare_macro_df(
    macro4,
    stage_filter=("data_collection", "control"),
    stages_csv=STAGES_CSV,
)
all_df4 = data_access.build_all_df(macro4)
all_df4 = all_df4[all_df4["site_label"] == "RewardSite"].copy()
print(f"Cohort fig4: {all_df4['mouse'].nunique()} mice, {all_df4['session_id'].nunique()} sessions")

# Cohort 2: fig6 (travel-cost manipulations)
EXCLUDED_FIG6 = {
    "707349", "713545", "713578", "715865", "715866", "715867",
    "715869", "715870", "716455", "716456", "716457", "716458",
}
STAGES_FIG6 = (
    "data_collection", "control",
    "friction", "friction_low", "friction_med", "friction_high",
    "distance_short", "distance_long", "distance_extra_short", "distance_extra_long",
)
_, _, macro6 = data_access._load_session_catalogue_and_sites(
    S3_ROOT, STORAGE_OPTIONS, EXCLUDED_FIG6
)
macro6 = data_access.prepare_macro_df(
    macro6, stage_filter=STAGES_FIG6, stages_csv=STAGES_CSV,
)
all_df6 = data_access.build_all_df(macro6)
all_df6 = all_df6[all_df6["site_label"] == "RewardSite"].copy()
print(f"Cohort fig6: {all_df6['mouse'].nunique()} mice, {all_df6['session_id'].nunique()} sessions")

# Cohort 3: fig7 (global reward manipulations)
EXCLUDED_FIG7 = {
    "754570", "754579", "754567", "754580", "754559", "754560", "754577",
    "754566", "754571", "754574", "754575", "754582", "745302", "745305", "745301",
}
STAGES_FIG7 = ("data_collection", "base", "control", "experiment1")
_, _, macro7 = data_access._load_session_catalogue_and_sites(
    S3_ROOT, STORAGE_OPTIONS, EXCLUDED_FIG7
)
macro7 = data_access.prepare_macro_df(
    macro7, stage_filter=STAGES_FIG7, stages_csv=STAGES_CSV, relabel_patches=False,
)
all_df7 = data_access.build_all_df(macro7)
all_df7 = all_df7[all_df7["site_label"] == "RewardSite"].copy()

# Remap odor names → tier labels for fig7
label_map = {
    "Alpha pinene": "60", "Alpha-pinene": "60",
    "Methyl Butyrate": "90", "Ethyl Butyrate": "90",
    "Amyl Acetate": "0", "Fenchone": "0",
}
all_df7["patch_label"] = all_df7["patch_label"].map(label_map).fillna(all_df7["patch_label"])
all_df7.loc[all_df7["stage_name"] == "base", "stage_name"] = "control"

# Build environment label per session (high/low based on max reward_probability in experiment1)
_manip1 = all_df7[
    (all_df7["stage_name"] == "experiment1") & (all_df7["patch_label"] != "0")
]
_sess_env = (
    _manip1.groupby("session_id")["reward_probability"].max().reset_index()
)
_sess_env.columns = ["session_id", "max_rp"]
_sess_env["environment"] = _sess_env["max_rp"].apply(
    lambda x: "high" if (pd.notna(x) and x > 0.6) else "low"
)
all_df7 = all_df7.merge(
    _sess_env[["session_id", "environment"]], on="session_id", how="left"
)
all_df7["environment"] = all_df7["environment"].fillna("mix")
print(f"Cohort fig7: {all_df7['mouse'].nunique()} mice, {all_df7['session_id'].nunique()} sessions")


## Panel A — MVT schematic

In [ ]:
# --- Panel A: MVT schematic ---
def plot_mvt_schematic(travel_time=1.5):
    patches = [
        dict(A=9.5, k=0.55, color=color1),
        dict(A=7.0, k=0.45, color="black"),
        dict(A=4.5, k=0.30, color=color2),
    ]

    def gain(t, A, k):
        return A * (1 - np.exp(-k * t))

    def gain_deriv(t, A, k):
        return A * k * np.exp(-k * t)

    p = patches[1]
    A, k = p["A"], p["k"]
    t_opt = brentq(
        lambda t: gain(t, A, k) / (t + travel_time) - gain_deriv(t, A, k), 0.1, 20
    )
    f_opt = gain(t_opt, A, k)
    slope = f_opt / (t_opt + travel_time)

    fig, ax = panel_fig("fig2_A")
    t_patch = np.linspace(0, t_opt + 3, 400)
    t_line = np.linspace(-travel_time, t_opt + 3, 400)

    y_max = max(pp["A"] for pp in patches) + 0.5
    ax.axvline(x=0, color="grey", linewidth=0.2, linestyle="--", zorder=0)
    ax.plot(t_line, slope * (t_line + travel_time), color="black", zorder=1, linewidth=0.8)
    for pp in patches:
        ax.plot(
            t_patch, gain(t_patch, pp["A"], pp["k"]),
            color=pp["color"], linewidth=0.8, zorder=2,
        )
    ax.plot(t_opt, f_opt, "ko", ms=2, zorder=5)
    ax.plot([t_opt, t_opt], [0, f_opt], "k--", zorder=3, linewidth=0.8)

    y_arrow = 6.0
    ax.annotate(
        "",
        xy=(0, y_arrow), xytext=(-travel_time, y_arrow),
        arrowprops=dict(arrowstyle="<->", color="grey"),
    )
    ax.text(-travel_time / 2, y_arrow + 0.5, "travel", ha="center", va="bottom", color="grey")
    ax.text(t_opt + 1.5, 1.6, "Leave", ha="center", va="top")

    ax.set_xlim(-travel_time - 0.3, t_opt + 3.2)
    ax.set_ylim(-1.2, y_max)
    ax.set_xlabel("Time", labelpad=12)
    ax.set_ylabel("Average reward intake")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["bottom"].set_position(("data", 0))
    ax.spines["left"].set_position(("data", -travel_time))
    ax.spines["left"].set_bounds(0, y_max)
    plt.tight_layout()
    return fig, ax

apply_style()
fig, ax = plot_mvt_schematic(travel_time=1.5)
save_panel(fig, PANELS_DIR / "fig2_A.pdf")
print("Saved fig2_A.pdf")


## Panel B — Reward depletion curves

In [ ]:
# --- Panel B: Reward depletion curves ---
apply_style()
fig, ax = panel_fig("fig2_B")

x = np.linspace(0, 10, 11)
b = math.e
c = 0.135

for a, col in [(0.9, color1), (0.6, color2), (0.0, color3)]:
    y = a * b ** (-c * x)
    ax.plot(x, y, color=col, marker="o", markersize=4)

ax.hlines(0.4, 0, 10, color="grey", linestyle="--")
ax.set_xlabel("Rewards collected")
ax.set_ylabel("P(reward)")
ax.set_ylim(-0.1, 1)
ax.set_xlim(-0.5, 10.5)
ax.set_yticks([0, 0.5, 1])
sns.despine(ax=ax)
plt.tight_layout()
save_panel(fig, PANELS_DIR / "fig2_B.pdf")
print("Saved fig2_B.pdf")


## Panel C — 6-metric comparison (2 × 3)

In [ ]:
# --- Panel C preparation: aggregate data for 6-metric comparison ---
groups = ["session_id", "mouse", "patch_index", "patch_label", "stage_name"]

pre_df4 = all_df4[
    (all_df4["last_visit"] == 1)
    & (all_df4["patch_index"] <= 50)
    & (all_df4["stage_name"].isin(["control", "data_collection"]))
]

mouse_df4 = (
    pre_df4.groupby(groups)
    .agg(
        site_number=("site_index_in_patch_by_type", "max"),
        reward_probability=("reward_probability", "min"),
        stops=("site_index_in_patch_by_type", "max"),
        total_rewards=("cumulative_rewards", "max"),
        consecutive_rewards=("consecutive_rewards", "max"),
        total_failures=("cumulative_failures", "max"),
        consecutive_failures=("consecutive_failures", "max"),
        time_patch=("exit_epoch", "max"),
    )
    .reset_index()
)

general_df4 = (
    mouse_df4.groupby(["mouse", "patch_label", "stage_name"])
    .agg(
        reward_probability=("reward_probability", "mean"),
        stops=("stops", "mean"),
        total_rewards=("total_rewards", "mean"),
        consecutive_failures=("consecutive_failures", "mean"),
        total_failures=("total_failures", "mean"),
        time_patch=("time_patch", "mean"),
    )
    .reset_index()
)
print(f"general_df4: {len(general_df4)} rows, {general_df4['mouse'].nunique()} mice")


In [ ]:
# --- Panel C: 6-metric comparison (2 × 3 multi-panel) ---
apply_style()
fig, axes = panel_fig("fig2_C", nrows=2, ncols=3)
axes = axes.flatten()

metric_cfg = [
    ("reward_probability", "P(reward)", (0.2, 0.8), np.arange(0.2, 0.81, 0.2)),
    ("stops",              "Stops",     (0, 20),     None),
    ("total_rewards",      "Rewards",   (0, 10),     None),
    ("consecutive_failures","Consec. omissions", (0, 3), None),
    ("total_failures",     "Total omissions", (0, 12.5), np.arange(0, 13, 5)),
    ("time_patch",         "Time in patch",   (0, 150),  np.arange(0, 151, 50)),
]

for ax, (var, title, ylim, yticks) in zip(axes, metric_cfg):
    plotting.plot_condition_comparison(
        general_df4,
        variable=var,
        condition="mouse",
        odor_labels=["90", "60"],
        color_dict=color_dict_label,
        ax=ax,
    )
    ax.set_title(title, loc="center")
    ax.set_xticks([])
    ax.set_ylabel("")
    ax.set_ylim(ylim)
    if yticks is not None:
        ax.set_yticks(yticks)

plt.tight_layout()
save_panel(fig, PANELS_DIR / "fig2_C.pdf")
print("Saved fig2_C.pdf")


## Panel D — P(leave) heatmap (per-mouse average)

In [ ]:
# --- Panel D: P(leave) heatmap (per-mouse average) ---
import matplotlib as mpl

max_number = 11
step = 5
vmax = 0.3
min_mice = 10

cmap_hm = mpl.colors.LinearSegmentedColormap.from_list(
    "YlGnBu_trunc", mpl.cm.get_cmap("YlGnBu")(np.linspace(0.05, 1.0, 256))
)
patch_meta = {"90": ("High", color1), "60": ("Low", color2)}

_patch4 = all_df4["patch_index"] < 50
hm_base = all_df4[_patch4]

fig, axes = panel_fig("fig2_D", ncols=2, sharex=True, sharey=True)
fig.subplots_adjust(wspace=0.08)
left_hm = None

for ax, patch_label in zip(axes.flatten(), ["90", "60"]):
    is_right = patch_label == "60"
    title_str, title_color = patch_meta[patch_label]

    _base = hm_base[hm_base["patch_label"] == patch_label]
    per_mouse = (
        _base.groupby(["mouse", "cumulative_rewards", "cumulative_failures"])
        ["has_choice"].mean()
        .reset_index()
    )
    mouse_count = (
        per_mouse.groupby(["cumulative_rewards", "cumulative_failures"])
        ["has_choice"].count()
        .unstack()
    )
    mouse_mean = (
        per_mouse.groupby(["cumulative_rewards", "cumulative_failures"])
        ["has_choice"].mean()
        .unstack()
    )

    plot = (1 - mouse_mean).astype(float)
    plot[mouse_count < min_mice] = float("nan")

    hm = sns.heatmap(
        plot, ax=ax, cmap=cmap_hm, cbar=True, vmax=vmax, vmin=0,
        cbar_kws={"fraction": 0.05, "shrink": 1.0},
    )
    if not is_right:
        left_hm = hm
    ax.invert_yaxis()
    ax.set_xlabel("Cumulative omissions")
    tick_vals = np.arange(0, max_number + 1, step)
    ax.set_xticks(tick_vals + 0.5, minor=False)
    ax.set_xticklabels([str(v) for v in tick_vals], rotation=0, ha="center")
    ax.set_yticks(tick_vals + 0.5, minor=False)
    ax.set_yticklabels([str(v) for v in tick_vals])
    ax.set_xlim(0, max_number)
    ax.set_ylim(0, max_number)
    ax.set_title(title_str, color=title_color)

    if is_right:
        ax.set_ylabel("")
        ax.tick_params(axis="y", left=False, labelleft=False)
        cbar = hm.collections[0].colorbar
        cbar.set_ticks([0, 0.1, 0.2, 0.3])
        cbar.set_ticklabels(["0", "0.1", "0.2", "0.3"])
        cbar.set_label("P(leave)", rotation=270, labelpad=8)
    else:
        ax.set_ylabel("Cumulative rewards")
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0, ha="right")

if left_hm is not None:
    left_hm.collections[0].colorbar.remove()

save_panel(fig, PANELS_DIR / "fig2_D.pdf")
print("Saved fig2_D.pdf")


## Panels E & F — Travel cost manipulations (fig6 cohort)

Aggregate and helper function shared by both panels.

In [ ]:
# --- Panels E & F preparation: travel-cost aggregation (fig6 cohort) ---
groups6 = ["session_id", "mouse", "patch_index", "patch_label", "stage_name"]

pre_df6 = all_df6[
    (all_df6["patch_label"].isin(["90", "60"]))
    & (all_df6["last_visit"] == 1)
    & (all_df6["site_index_in_patch_by_type"] > 0)
    & (all_df6["patch_index"] <= 50)
]

mouse_df6 = (
    pre_df6.groupby(groups6)
    .agg(
        reward_probability=("reward_probability", "min"),
        stops=("site_index_in_patch_by_type", "max"),
        total_rewards=("cumulative_rewards", "max"),
        consecutive_failures=("consecutive_failures", "max"),
        total_failures=("cumulative_failures", "max"),
    )
    .reset_index()
)

general_df6 = (
    mouse_df6.groupby(["mouse", "stage_name"])
    .agg(
        reward_probability=("reward_probability", "mean"),
        stops=("stops", "mean"),
        total_rewards=("total_rewards", "mean"),
        consecutive_failures=("consecutive_failures", "mean"),
        total_failures=("total_failures", "mean"),
    )
    .reset_index()
    .rename(columns={"stage_name": "experiment"})
)
print(f"general_df6: {len(general_df6)} rows, {general_df6['mouse'].nunique()} mice")


In [ ]:
# Helper: plot travel-cost manipulation (used by panels E and F)
def plot_travel_cost(
    general_df,
    variable="reward_probability",
    experiment_variable="distance",  # "distance" or "friction"
    mean="control",
    panel_key=None,
):
    labels_dist = {"50": "50", "150": "150", "300": "300", "450": "450", "600": "600"}

    if experiment_variable == "friction":
        experiments = ["control", "friction_low", "friction_med", "friction_high"]
        xlabel = "Added inter-patch friction (a.u.)"
        labels_plot = ["None", "Low", "Medium", "High"]
        _blues = plt.cm.Blues(np.linspace(0.4, 0.9, 3))
        condition_colors = {
            "control": "#888888",
            "friction_low": _blues[0],
            "friction_med": _blues[1],
            "friction_high": _blues[2],
        }
    else:
        experiments = [
            "distance_extra_short", "distance_short", "control",
            "distance_long", "distance_extra_long",
        ]
        xlabel = "Inter-patch travel distance (cm)"
        labels_plot = ["50", "150", "300", "450", "600"]
        _reds = plt.cm.Reds(np.linspace(0.4, 0.9, 4))
        condition_colors = {
            "control": "#888888",
            "distance_extra_short": _reds[0],
            "distance_short": _reds[1],
            "distance_long": _reds[2],
            "distance_extra_long": _reds[3],
        }

    norm_var = variable + "_norm"
    summary = general_df.copy()
    summary["mouse"] = summary["mouse"].astype(str)
    ref_vals = summary.loc[summary["experiment"] == mean, ["mouse", variable]].set_index("mouse")[variable]
    summary[norm_var] = summary[variable] - summary["mouse"].map(ref_vals)

    if panel_key:
        fig, ax = panel_fig(panel_key)
    else:
        fig, ax = plt.subplots(1, 1, figsize=(4.75, 4.25))

    x_map = {exp: i for i, exp in enumerate(experiments)}
    for mouse, mdf in summary[summary["experiment"].isin(experiments)].groupby("mouse"):
        pts = mdf.set_index("experiment")[norm_var].reindex(experiments).dropna()
        if len(pts) >= 2:
            ax.plot(
                [x_map[e] for e in pts.index], pts.values,
                color="gray", alpha=0.4, linewidth=0.5,
            )

    for exp in experiments:
        subset = summary.loc[summary["experiment"] == exp, norm_var].dropna()
        if subset.empty:
            continue
        c = condition_colors.get(exp, "#444444")
        ax.errorbar(
            x_map[exp], subset.mean(), yerr=subset.std(),
            fmt="o", color=c,
            capsize=0, capthick=0, elinewidth=1, markersize=5, zorder=10,
        )

    ax.axhline(0, color="black", linestyle="--", alpha=1, linewidth=1)
    ax.set_xticks(list(x_map.values()), labels_plot)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    # Significance vs reference (paired t-test)
    ref_v = summary.loc[summary["experiment"] == mean].set_index("mouse")[norm_var]
    y_top = ax.get_ylim()[1]
    for i, exp in enumerate(experiments):
        if exp == mean:
            continue
        exp_v = summary.loc[summary["experiment"] == exp].set_index("mouse")[norm_var]
        paired = pd.concat([ref_v, exp_v], axis=1, join="inner").dropna()
        paired.columns = ["ref", "exp"]
        if len(paired) < 3:
            continue
        try:
            _, p = ttest_rel(paired["ref"], paired["exp"])
        except Exception:
            continue
        sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
        ax.text(i, y_top, sig, ha="center", va="top", fontsize=7)

    ax.set_ylabel(
        "\u0394P(reward) at leaving" if variable == "reward_probability"
        else f"\u0394{variable}"
    )
    ax.set_xlabel(xlabel)
    sns.despine(ax=ax)
    plt.tight_layout()
    return fig, ax


## Panel E — Distance manipulation

In [ ]:
# --- Panel E: Distance manipulation ---
fig, ax = plot_travel_cost(
    general_df6,
    variable="reward_probability",
    experiment_variable="distance",
    mean="control",
    panel_key="fig2_E",
)
save_panel(fig, PANELS_DIR / "fig2_E.pdf")
print("Saved fig2_E.pdf")


## Panel F — Friction manipulation

In [ ]:
# --- Panel F: Friction manipulation ---
fig, ax = plot_travel_cost(
    general_df6,
    variable="reward_probability",
    experiment_variable="friction",
    mean="control",
    panel_key="fig2_F",
)
save_panel(fig, PANELS_DIR / "fig2_F.pdf")
print("Saved fig2_F.pdf")


## Panels G, H, I — Global reward manipulations (fig7 cohort)

Aggregate and helper function shared by panels H and I.

In [ ]:
# --- Panels G, H, I preparation: global reward aggregation (fig7 cohort) ---
groups7 = ["session_id", "mouse", "patch_index", "patch_label", "stage_name", "environment"]

pre_df7 = all_df7[
    (all_df7["patch_label"] != "0")
    & (all_df7["last_visit"] == 1)
    & (all_df7["patch_index"] <= 50)
]

mouse_df7 = (
    pre_df7.groupby(groups7)
    .agg(
        reward_probability=("reward_probability", "min"),
        stops=("site_index_in_patch_by_type", "max"),
        total_rewards=("cumulative_rewards", "max"),
        consecutive_failures=("consecutive_failures", "max"),
        total_failures=("cumulative_failures", "max"),
    )
    .reset_index()
    .rename(columns={"stage_name": "experiment"})
)

general_df7 = (
    mouse_df7.groupby(["mouse", "patch_label", "experiment", "environment"])
    .agg(
        reward_probability=("reward_probability", "mean"),
        stops=("stops", "mean"),
        total_rewards=("total_rewards", "mean"),
        consecutive_failures=("consecutive_failures", "mean"),
        total_failures=("total_failures", "mean"),
    )
    .reset_index()
)
print(f"general_df7: {len(general_df7)} rows, {general_df7['mouse'].nunique()} mice")


In [ ]:
# Helper: plot change in reward probability between control and experiment1
def plot_reward_diff(general_df, environment="high", variable="reward_probability", ax=None):
    summary = general_df.copy()
    mouse_list = summary.loc[summary["environment"] == environment, "mouse"].unique()
    patch_order = ["90", "60"]
    patch_colors = {"60": color2, "90": color1}
    x_map = {pl: i for i, pl in enumerate(patch_order)}

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(4.25, 4.25))
    else:
        fig = ax.get_figure()

    all_diffs = []
    for pl in patch_order:
        base = (
            summary.loc[
                (summary["experiment"] == "control")
                & (summary["environment"] == "mix")
                & (summary["mouse"].isin(mouse_list))
                & (summary["patch_label"] == pl)
            ][["mouse", variable]]
            .rename(columns={variable: "base"})
        )
        manip = (
            summary.loc[
                (summary["experiment"] == "experiment1")
                & (summary["environment"] == environment)
                & (summary["mouse"].isin(mouse_list))
                & (summary["patch_label"] == pl)
            ][["mouse", variable]]
            .rename(columns={variable: "manip"})
        )
        diff_df = base.merge(manip, on="mouse")
        diff_df["difference"] = diff_df["manip"] - diff_df["base"]
        diff_df["patch_label"] = pl
        all_diffs.append(diff_df)

    combined = pd.concat(all_diffs, ignore_index=True)

    # Gray connecting lines per mouse
    pivot = combined.pivot(index="mouse", columns="patch_label", values="difference")
    for _, row in pivot.iterrows():
        vals = row.reindex(patch_order).dropna()
        if len(vals) == 2:
            ax.plot([x_map[pl] for pl in vals.index], vals.values,
                    color="gray", alpha=0.4, linewidth=0.8)

    # Mean +/- SD dots
    for pl in patch_order:
        vals = combined.loc[combined["patch_label"] == pl, "difference"].dropna()
        ax.errorbar(
            x_map[pl], vals.mean(), yerr=vals.std(),
            fmt="o", color=patch_colors[pl],
            capsize=0, capthick=0, elinewidth=1.5, markersize=7, zorder=10,
        )

    ax.set_xlim(-0.5, 1.5)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1, zorder=5)
    ax.set_xticks(list(x_map.values()), ["Patch 90", "Patch 60"])

    # Significance: paired t-test between patch types
    p60 = combined.loc[combined["patch_label"] == "60"].set_index("mouse")["difference"]
    p90 = combined.loc[combined["patch_label"] == "90"].set_index("mouse")["difference"]
    paired = pd.concat([p60, p90], axis=1, join="inner").dropna()
    paired.columns = ["p60", "p90"]
    if len(paired) >= 3:
        _, p = ttest_rel(paired["p60"], paired["p90"])
        sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
        y_lo, y_hi = ax.get_ylim()
        pad = (y_hi - y_lo) * 0.06
        y_bracket = combined["difference"].max() + pad
        ax.set_ylim(top=y_bracket + pad)
        ax.plot([0, 1], [y_bracket, y_bracket], color="black", linewidth=0.8)
        ax.text(0.5, y_bracket + pad * 0.3, sig, ha="center", va="bottom", fontsize=7)

    ylabel_map = {
        "reward_probability": "\u0394P(reward) at patch leaving",
        "total_rewards": "\u0394Rewards",
        "stops": "\u0394Stops",
    }
    ax.set_ylabel(ylabel_map.get(variable, f"\u0394{variable}"))
    sns.despine(ax=ax)
    return fig, ax


## Panel G — Transition schematics

In [ ]:
# --- Panel G: Global-reward transition schematics ---
apply_style()
fig, axes = panel_fig("fig2_G", ncols=2)
ax1, ax2 = axes

max_x = 10
b = math.e
c = 0.1284
marker = "o"
x = np.linspace(0, max_x, max_x)

# Left subplot: Low -> High transition
a = 0.9
ax1.plot(x, a * b ** (-c * x), color=color1, marker=marker, markersize=3)
a = 0.6
ax1.plot(x, a * b ** (-c * x), color=color2, marker=marker, alpha=0.5, markersize=4, markeredgewidth=0)
x_shift, y_shift = 0.1, 0.005
x_s = np.linspace(x_shift, max_x, max_x)
a = 0.9
ax1.plot(x_s, a * b ** (-c * (x_s - x_shift)) + y_shift, color=color2, marker=marker, alpha=1, markersize=3)
ax1.annotate("", xy=(0, 0.88), xytext=(0, 0.62),
             arrowprops=dict(arrowstyle="->", color="red", lw=1.5))
ax1.set_xlabel("Rewards collected")
ax1.set_ylabel("P(reward)")
ax1.set_ylim(-0.1, 1)
ax1.set_xlim(-0.5, max_x + 0.5)
ax1.yaxis.set_major_locator(FixedLocator([0, 0.5, 1]))
ax1.xaxis.set_major_locator(FixedLocator(np.arange(0, max_x + 1, 5)))
ax1.set_title("Low → High", fontsize=6)

# Right subplot: High -> Low transition
a = 0.6
ax2.plot(x_s, a * b ** (-c * (x_s - x_shift)) + y_shift, color=color1, marker=marker, markersize=3, zorder=10)
a = 0.6
ax2.plot(x, a * b ** (-c * x), color=color2, marker=marker, markersize=3)
a = 0.9
ax2.plot(x, a * b ** (-c * x), color=color1, marker=marker, alpha=0.5, markersize=4, markeredgewidth=0)
ax2.annotate("", xy=(0.1, 0.86), xytext=(0.1, 0.62),
             arrowprops=dict(arrowstyle="<-", color="red", lw=1.5))
ax2.set_ylabel("P(reward)")
ax2.set_xlabel("Rewards collected")
ax2.set_ylim(-0.1, 1)
ax2.set_xlim(-0.5, max_x + 0.5)
ax2.yaxis.set_major_locator(FixedLocator([0, 0.5, 1]))
ax2.xaxis.set_major_locator(FixedLocator(np.arange(0, max_x + 1, 5)))
ax2.set_title("High → Low", fontsize=6)

sns.despine()
plt.tight_layout()
save_panel(fig, PANELS_DIR / "fig2_G.pdf")
print("Saved fig2_G.pdf")


## Panel H — Higher global reward

In [ ]:
# --- Panel H: Higher global reward results ---
apply_style()
fig, axs = panel_fig("fig2_H", ncols=2)
plot_reward_diff(general_df7, environment="high", variable="reward_probability", ax=axs[0])
axs[0].set_ylim(-0.2, 0.2)
plot_reward_diff(general_df7, environment="high", variable="total_rewards", ax=axs[1])
axs[1].set_ylim(-4, 4)
fig.suptitle("Higher global reward", fontsize=7)
plt.tight_layout(w_pad=2, pad=0.5)
save_panel(fig, PANELS_DIR / "fig2_H.pdf")
print("Saved fig2_H.pdf")


## Panel I — Lower global reward

In [ ]:
# --- Panel I: Lower global reward results ---
apply_style()
fig, axs = panel_fig("fig2_I", ncols=2)
plot_reward_diff(general_df7, environment="low", variable="reward_probability", ax=axs[0])
axs[0].set_ylim(-0.2, 0.2)
plot_reward_diff(general_df7, environment="low", variable="total_rewards", ax=axs[1])
axs[1].set_ylim(-4, 4)
fig.suptitle("Lower global reward", fontsize=7)
plt.tight_layout(w_pad=2, pad=0.5)
save_panel(fig, PANELS_DIR / "fig2_I.pdf")
print("Saved fig2_I.pdf")
